# T3: TrOCR fine-tune on handwritten medicine-name images

**Run on Kaggle or Colab with a T4 GPU** (about 30 to 60 minutes).

**Data.** RxHandBD (Mendeley Data, 5,578 images of handwritten medicine names) or the Doctor's Handwritten Prescription BD set. Download it yourself and read its licence before using it; this notebook does not redistribute it. Add it as a Kaggle dataset or upload to Colab.

**Unverified assumption.** The folder layout of the download was not checked when this notebook was written. Cell 3 detects either a CSV (image, text) or one folder per word, prints what it found, and stops if it looks wrong. Edit only that cell.

**Split.** Random 80/10/10 by image. The data has no writer identifiers that I could verify, so this does **not** test unseen writers. Do not claim that.

In [ ]:
!pip -q install -U transformers datasets accelerate jiwer pillow

In [ ]:
import glob, os, random, time, json
import torch, pandas as pd
from PIL import Image
ROOT = os.environ.get('RXHAND', '/kaggle/input')   # folder that holds the downloaded dataset
imgs = [p for p in glob.glob(ROOT + '/**/*', recursive=True) if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(len(imgs), 'images under', ROOT); print(imgs[:3])

In [ ]:
samples = []   # (path, text)
csvs = glob.glob(ROOT + '/**/*.csv', recursive=True)
by_name = {os.path.basename(p): p for p in imgs}
for c in csvs:
    df = pd.read_csv(c)
    img_col = next((k for k in df.columns if df[k].astype(str).str.lower().str.endswith(('.png', '.jpg', '.jpeg')).mean() > .8), None)
    txt_col = next((k for k in df.columns if k != img_col and df[k].dtype == object), None)
    if img_col and txt_col:
        samples += [(by_name[os.path.basename(r[img_col])], str(r[txt_col]).strip()) for _, r in df.iterrows() if os.path.basename(r[img_col]) in by_name]
if not samples:   # fall back to one folder per word
    samples = [(p, os.path.basename(os.path.dirname(p))) for p in imgs]
print(len(samples), 'labelled images'); print(samples[:5])
assert len(samples) > 1000 and len({t for _, t in samples}) > 20, 'labels look wrong: edit this cell for your folder layout'

In [ ]:
random.seed(0); random.shuffle(samples)
n = len(samples); tr, va, te = samples[:int(.8 * n)], samples[int(.8 * n):int(.9 * n)], samples[int(.9 * n):]
print(len(tr), len(va), len(te))
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
NAME = 'microsoft/trocr-base-handwritten'
proc = TrOCRProcessor.from_pretrained(NAME)
model = VisionEncoderDecoderModel.from_pretrained(NAME).to('cuda')
model.config.decoder_start_token_id = proc.tokenizer.cls_token_id; model.config.pad_token_id = proc.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

In [ ]:
import jiwer
@torch.no_grad()
def cer(model, rows, batch=32):
    model.eval(); preds, refs = [], []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        px = proc([Image.open(p).convert('RGB') for p, _ in chunk], return_tensors='pt').pixel_values.to('cuda')
        out = model.generate(px, max_new_tokens=24)
        preds += proc.batch_decode(out, skip_special_tokens=True); refs += [t for _, t in chunk]
    return round(jiwer.cer(refs, preds), 4), list(zip(refs[:8], preds[:8]))

before, ex_before = cer(model, te); print('CER before', before); print(ex_before)

In [ ]:
class DS(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        p, t = self.rows[i]
        px = proc(Image.open(p).convert('RGB'), return_tensors='pt').pixel_values[0]
        ids = proc.tokenizer(t, padding='max_length', max_length=24, truncation=True).input_ids
        return {'pixel_values': px, 'labels': torch.tensor([x if x != proc.tokenizer.pad_token_id else -100 for x in ids])}

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
args = Seq2SeqTrainingArguments(output_dir='t3_out', per_device_train_batch_size=16, num_train_epochs=4, learning_rate=4e-5,
    fp16=True, logging_steps=50, save_strategy='no', report_to='none', remove_unused_columns=False, predict_with_generate=False)
tr_ = Seq2SeqTrainer(model=model, args=args, train_dataset=DS(tr), data_collator=default_data_collator)
t = time.time(); tr_.train(); train_s = round(time.time() - t); print('trained in', train_s, 's')

In [ ]:
after, ex_after = cer(model, te); print('CER after', after); print(ex_after)
res = {'model': NAME, 'train_images': len(tr), 'test_images': len(te), 'split': 'random by image (no writer ids)',
       'cer_before': before, 'cer_after': after, 'train_seconds': train_s}
json.dump(res, open('t3_results.json', 'w'), indent=1); model.save_pretrained('t3_trocr'); proc.save_pretrained('t3_trocr')
!zip -qr t3_trocr.zip t3_trocr
print(res)

## Reading the result

Character error rate is the share of characters wrong, so lower is better. The test images share writers with the training images, so this shows the model learned these medicine words, not that it reads any doctor's hand. Report it that way.